In [1]:
!pip install torch_geometric
!pip install spektral


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 3.0 MB/s eta 0:00:00


In [2]:
!kaggle datasets download -d itsanmol124/mimic-cxr

Dataset URL: https://www.kaggle.com/datasets/itsanmol124/mimic-cxr
License(s): unknown
 99% 2.03G/2.04G [00:27<00:00, 116MB/s]
100% 2.04G/2.04G [00:27<00:00, 79.4MB/s]


In [4]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
import tensorflow.sparse as tf_sparse
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, concatenate, Lambda
from spektral.utils import normalized_adjacency
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras import layers, models, metrics, regularizers, callbacks
from spektral.layers import GINConv
from sklearn.metrics.pairwise import cosine_similarity

# Unzip the dataset
import zipfile
with zipfile.ZipFile("/content/mimic-cxr.zip", "r") as zip_ref:
    zip_ref.extractall("/content/mimic_cxr")
print("Dataset extracted to /content/mimic_cxr")

# Load CSV file
extract_dir = "/content/mimic_cxr"
csv_file = os.path.join(extract_dir, "mimic-cxr.csv")

data_df = pd.read_csv(csv_file)

# Define tabular features
tabular_features = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
    "Enlarged Cardiomediastinum", "Lung Lesion", "Lung Opacity",
    "Pleural Effusion", "Pneumonia", "Pneumothorax"
]

# Convert tabular features to float32
data_df[tabular_features] = data_df[tabular_features].astype(np.float32)

# Process labels
def process_labels(label_str):
    all_labels = [
        "Normal", "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
        "Enlarged Cardiomediastinum", "Lung Lesion", "Lung Opacity",
        "Pleural Effusion", "Pneumonia", "Pneumothorax"
    ]
    labels = label_str.split(", ")
    return [1 if label in labels else 0 for label in all_labels]

data_df['label'] = data_df['label'].apply(process_labels)

# Add file paths
data_df['filepath'] = data_df.apply(lambda row: os.path.join(extract_dir, row['split'], row['filename']), axis=1)

# Split data into train, validation, and test sets
train_data = data_df[data_df['split'] == 'train'].sample(n=1000, random_state=42)
valid_data = data_df[data_df['split'] == 'valid'].reset_index(drop=True)
test_data = data_df[data_df['split'] == 'test'].reset_index(drop=True)

# Preprocess images
def preprocess_image(image_path):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [224, 224])
    image /= 255.0
    return image

# Preprocess tabular data
def preprocess_tabular_data(tabular_values):
    return tf.convert_to_tensor(tabular_values, dtype=tf.float32)

# Load and preprocess data
def load_and_preprocess(filepath, label, tabular_values):
    image = preprocess_image(filepath)
    tabular_data = preprocess_tabular_data(tabular_values)
    label = tf.cast(label, dtype=tf.float32)
    return (image, tabular_data), label

# Create TensorFlow datasets
def create_tf_dataset(data):
    filepaths = data["filepath"].tolist()
    labels = np.array(data["label"].tolist())
    tabular_data = data[tabular_features].values.astype(np.float32)
    dataset = tf.data.Dataset.from_tensor_slices((filepaths, labels, tabular_data))
    dataset = dataset.map(lambda filepath, label, tabular_values: load_and_preprocess(filepath, label, tabular_values),
                          num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.batch(32).prefetch(buffer_size=tf.data.AUTOTUNE)

train_dataset = create_tf_dataset(train_data)
valid_dataset = create_tf_dataset(valid_data)
test_dataset = create_tf_dataset(test_data)

# Custom GINConv layer to handle model correctly
class CustomGINConv(GINConv):
    def call(self, inputs):
        x, a = inputs  # x: node features (batch, nodes, features), a: adjacency matrix (nodes, nodes)
        return super().call([x, a])

# Layer to provide adjacency matrix
class AdjacencyLayer(layers.Layer):
    def __init__(self, adj_matrix, **kwargs):
        super().__init__(**kwargs)
        self.adj_matrix = adj_matrix

    def call(self, inputs):
        return self.adj_matrix

# Build hybrid GIN model
def build_hybrid_gin_model(num_features, adj_matrix):
    # Image input branch
    image_input = Input(shape=(224, 224, 3), name="image_input")
    base_model = tf.keras.applications.ResNet50(weights="imagenet", include_top=False, input_tensor=image_input)
    image_features = GlobalAveragePooling2D()(base_model.output)

    # Tabular input branch
    tabular_input = Input(shape=(None, num_features), name="tabular_input")  # (batch, nodes, features)

    # Add adjacency matrix
    adj_layer = AdjacencyLayer(adj_matrix)(tabular_input)  # Outputs (nodes, nodes)

    # GINConv layer
    gin_output = CustomGINConv(channels=16)([tabular_input, adj_layer])
    gin_output = layers.GlobalAveragePooling1D()(gin_output)

    # Combine features
    combined = concatenate([image_features, gin_output])
    output = Dense(11, activation="sigmoid")(combined)

    model = models.Model(inputs=[image_input, tabular_input], outputs=output)
    return model

# Construct adjacency matrix using cosine similarity
tabular_data = train_data[tabular_features].values
adj_matrix = cosine_similarity(tabular_data)
adj_matrix = normalized_adjacency(adj_matrix)

# Convert to sparse tensor
indices = np.array(np.nonzero(adj_matrix)).T
values = adj_matrix[indices[:, 0], indices[:, 1]]
adj_sparse = tf_sparse.SparseTensor(indices=indices, values=values.astype(np.float32), dense_shape=adj_matrix.shape)
adj_sparse = tf.sparse.reorder(adj_sparse)

# Build the model
num_features = len(tabular_features)
hybrid_gin_model = build_hybrid_gin_model(num_features, adj_sparse)
hybrid_gin_model.summary()

# Compile the model
hybrid_gin_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", Precision(name="precision"), Recall(name="recall")]
)

# Prepare training data
train_images = np.array([preprocess_image(filepath) for filepath in train_data['filepath']])
train_tabular = np.repeat(train_data[tabular_features].values[:, np.newaxis, :], adj_matrix.shape[0], axis=1)
train_labels = np.array(train_data['label'].tolist())

# Prepare validation data
val_images = np.array([preprocess_image(filepath) for filepath in valid_data['filepath']])
val_tabular = np.repeat(valid_data[tabular_features].values[:, np.newaxis, :], adj_matrix.shape[0], axis=1)
val_labels = np.array(valid_data['label'].tolist())

# Train the model
history = hybrid_gin_model.fit(
    x=[train_images, train_tabular],
    y=train_labels,
    batch_size=16,
    epochs=25,
    validation_data=([val_images, val_tabular], val_labels)
)

# Calculate metrics
val_precision = history.history['val_precision'][-1]
val_recall = history.history['val_recall'][-1]
val_f1 = 2 * (val_precision * val_recall) / (val_precision + val_recall + 1e-7)
print(f"Validation Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}")

# Prepare test data
test_images = np.array([preprocess_image(filepath) for filepath in test_data['filepath']])
test_tabular = np.repeat(test_data[tabular_features].values[:, np.newaxis, :], adj_matrix.shape[0], axis=1)
test_labels = np.array(test_data['label'].tolist())

# Evaluate
test_results = hybrid_gin_model.evaluate(
    x=[test_images, test_tabular],
    y=test_labels,
    batch_size=16,
    verbose=1
)
test_loss, test_acc, test_prec, test_rec = test_results
test_f1 = 2 * (test_prec * test_rec) / (test_prec + test_rec + 1e-7)
print(f"Test Accuracy: {test_acc:.4f}, Precision: {test_prec:.4f}, Recall: {test_rec:.4f}, F1: {test_f1:.4f}")

Dataset extracted to /content/mimic_cxr


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ image_input (InputLayer)  │ (None, 224, 224, 3)    │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1_pad (ZeroPadding2D) │ (None, 230, 230, 3)    │              0 │ image_input[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1_conv (Conv2D)       │ (None, 112, 112, 64)   │          9,472 │ conv1_pad[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1_bn                  │ (None, 112, 112, 64)   │            256 │ conv1_conv[0][0]       │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1_relu (Activation)   │ (None, 112, 112, 64)   │              0 │ conv1_bn[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ pool1_pad (ZeroPadding2D) │ (None, 114, 114, 64)   │              0 │ conv1_relu[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ pool1_pool (MaxPooling2D) │ (None, 56, 56, 64)     │              0 │ pool1_pad[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_1_conv       │ (None, 56, 56, 64)     │          4,160 │ pool1_pool[0][0]       │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_1_bn         │ (None, 56, 56, 64)     │            256 │ conv2_block1_1_conv[0… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_1_relu       │ (None, 56, 56, 64)     │              0 │ conv2_block1_1_bn[0][… │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_2_conv       │ (None, 56, 56, 64)     │         36,928 │ conv2_block1_1_relu[0… │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_2_bn         │ (None, 56, 56, 64)     │            256 │ conv2_block1_2_conv[0… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_2_relu       │ (None, 56, 56, 64)     │              0 │ conv2_block1_2_bn[0][… │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_0_conv       │ (None, 56, 56, 256)    │         16,640 │ pool1_pool[0][0]       │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_3_conv       │ (None, 56, 56, 256)    │         16,640 │ conv2_block1_2_relu[0… │
│ (Conv2D)                  │                        │                │                        │
├──────────────────────

 Total params: 23,610,604 (90.07 MB)

 Trainable params: 23,557,484 (89.86 MB)

 Non-trainable params: 53,120 (207.50 KB)

Epoch 1/25
63/63 ━━━━━━━━━━━━━━━━━━━━ 110s 775ms/step - accuracy: 0.4868 - loss: 1.1603 - precision: 0.4331 - recall: 0.5014 - val_accuracy: 0.1603 - val_loss: 30.9342 - val_precision: 0.1483 - val_recall: 0.3665
Epoch 2/25
63/63 ━━━━━━━━━━━━━━━━━━━━ 13s 207ms/step - accuracy: 0.8081 - loss: 0.0521 - precision: 0.9577 - recall: 0.9611 - val_accuracy: 0.8776 - val_loss: 0.1242 - val_precision: 0.8018 - val_recall: 0.9956
Epoch 3/25
63/63 ━━━━━━━━━━━━━━━━━━━━ 20s 202ms/step - accuracy: 0.8872 - loss: 0.0136 - precision: 0.9861 - recall: 0.9948 - val_accuracy: 0.8368 - val_loss: 0.0485 - val_precision: 0.9250 - val_recall: 0.9895
Epoch 4/25
63/63 ━━━━━━━━━━━━━━━━━━━━ 21s 203ms/step - accuracy: 0.8708 - loss: 0.0058 - precision: 0.9971 - recall: 1.0000 - val_accuracy: 0.7300 - val_loss: 0.0999 - val_precision: 0.8868 - val_recall: 0.8682
Epoch 5/25
63/63 ━━━━━━━━━━━━━━━━━━━━ 21s 207ms/step - accuracy: 0.8819 - loss: 0.0031 - precision: 0.9995 - recall: 1.0000 - val_accuracy: 0.6667 - val_l